# Task 3 — Kafka topics and event contract

```{admonition} Evidence summary
:class: note

| Field | Value |
|---|---|
| Evidence source | Sanitized Kafka samples hashed by `stage2_manifest.json` |
| Commands | `bash scripts/init_kafka_topics.sh`; `bash scripts/capture_kafka_evidence.sh` |
| Topics | `cpg.nodes`, `cpg.edges`, `cpg.metadata`, `cpg.errors` |
| Required fields | `schema_version`, `event_time`, and complete provenance |
```

## Approach and reasoning

Separate topics let Neo4j subscribe only to graph topology while Spark subscribes
only to metadata. All four contracts share version, UTC event time, repository,
commit, run, file identity, and file path fields.

Raw artifacts:
[nodes](../screenshots/kafka/sample_cpg_nodes.json),
[edges](../screenshots/kafka/sample_cpg_edges.json),
[metadata](../screenshots/kafka/sample_cpg_metadata.json),
[errors](../screenshots/kafka/sample_cpg_errors.json).

## Key implementation excerpts

```{literalinclude} ../scripts/init_kafka_topics.sh
:language: bash
:lines: 1-28
:caption: Explicit four-topic layout
```

```{literalinclude} ../parser_service/schemas.py
:language: python
:pyobject: common_fields
:caption: Shared schema version, event time, and provenance fields
```

## Executed evidence


In [1]:
from pathlib import Path
import json

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "screenshots/stage2_manifest.json").exists()
)
sample_files = {
    "cpg.nodes": "sample_cpg_nodes.json",
    "cpg.edges": "sample_cpg_edges.json",
    "cpg.metadata": "sample_cpg_metadata.json",
    "cpg.errors": "sample_cpg_errors.json",
}
required = {"schema_version", "event_time", "repo", "commit_sha", "run_id", "file_id", "file_path"}

for topic, filename in sample_files.items():
    path = ROOT / "screenshots/kafka" / filename
    message = json.loads(next(line for line in path.read_text().splitlines() if line.strip()))
    print(f"{topic} sample:")
    print(json.dumps(message, sort_keys=True))
    assert required <= message.keys()
    assert message["schema_version"] == "1.0"
    assert message["event_time"].endswith("Z")

topic_details = (ROOT / "screenshots/kafka/topic_details.txt").read_text()
assert all(topic in topic_details for topic in sample_files)


cpg.nodes sample:
{"col_offset": null, "commit_sha": "41adfd0f9ee9ba3a6b4f719d5b551c5b19ae45e2", "end_col_offset": null, "end_lineno": null, "event_time": "2026-07-23T06:58:02.430Z", "file_id": "6c39568a6a11c430", "file_path": "src/datasets/__init__.py", "id": "57d761e9567d5b95", "lineno": null, "node_type": "Module", "op": "upsert", "properties": {}, "repo": "huggingface/datasets", "run_id": "8c3efe458b3b43eaa1399d7740848d3f", "schema_version": "1.0", "scope_path": "<module>"}
cpg.edges sample:
{"commit_sha": "41adfd0f9ee9ba3a6b4f719d5b551c5b19ae45e2", "edge_type": "CFG_NEXT", "event_time": "2026-07-23T06:58:11.913Z", "file_id": "97d05a67e514a1e5", "file_path": "src/datasets/builder.py", "id": "07474daf0400bc31", "op": "upsert", "properties": {"extractor": "cfg"}, "repo": "huggingface/datasets", "run_id": "8c3efe458b3b43eaa1399d7740848d3f", "schema_version": "1.0", "source_id": "e27b96489764bc8f", "target_id": "3c49b372d1e08d59"}
cpg.metadata sample:
{"commit_sha": "41adfd0f9ee9ba3a6b

## What this chapter proves

| Requirement | Evidence |
|---|---|
| Four topic categories | The topic creation source and topic-details artifact contain all four names. |
| Real messages | The executed output prints one captured JSON message per topic. |
| Forward compatibility | Every sample asserts `schema_version=1.0`. |
| Event time | Every sample asserts a UTC timestamp ending in `Z`. |

## Reflection

```{admonition} Closing reflection
:class: tip

| Point | Result |
|---|---|
| Worked | Topic ownership cleanly separates graph, metadata, and errors. |
| Failed | The old chapter printed only aggregate counts, not message bodies. |
| Resolution | The chapter now renders four real hash-validated JSON samples. |
```
